# What happens to a feature vector inside LogiX-GIN + prototypes

This notebook walks **one batch of molecules** through a trained model and shows the
representation at every stage, so you can see what each step does to the numbers.

The pipeline, stage by stage:

| # | stage | shape | what it is |
|---|-------|-------|------------|
| 0 | raw input `x` | `[N, 14]` | atom one-hot per node |
| 1 | doubled input `[x, 1-x]` | `[N, 28]` | every literal beside its negation |
| 2 | GINConv aggregate | `[N, 28]` | **counts** over the closed neighbourhood |
| 3 | `phi_in` argument | `[N, 28]` | `w * count + b`, input to the step |
| 4 | literals after `phi_in` | `[N, 28]` | thresholded -> truth values |
| 5 | layer output `h_l` | `[N, 64]` | one GML predicate per dimension |
| 6 | jumping-knowledge concat | `[N, 192]` | 3 layers x 64, radius 1/2/3 |
| 7 | pooled `z` | `[G, 384]` | mean &#124; max over nodes |
| 8 | similarity `s` | `[G, 16]` | Hamming agreement with each prototype |
| 9 | head literals `[s, 1-s]` -> logits | `[G, 32]` -> `[G, 2]` | the logic head |

The two places where interpretability is won or lost are **stage 3** (is the step a
threshold or has it wrapped?) and **stage 7** (pooling turns truth values into
fractions). Both get their own section.

## Setup

In [ ]:
import os, sys, glob, json, pickle, math
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

REPO = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'nbs' else os.path.abspath('.')
sys.path.insert(0, REPO); os.chdir(REPO)

from torch_geometric.loader import DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ATOMS  = ['C','O','Cl','H','N','F','Br','S','P','I','Na','K','Li','Ca']
print('repo:', REPO, '| device:', DEVICE)

In [ ]:
# --- plot style: recessive axes, ink-coloured text, fixed categorical order ---
SERIES = ['#2a78d6', '#eb6834', '#1baf7a']          # blue, orange, aqua (validated all-pairs)
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#8a8981'
SURFACE = '#fcfcfb'
BLUES = ['#cde2fb','#9ec5f4','#6da7ec','#3987e5','#256abf','#184f95','#0d366b']
SEQ = LinearSegmentedColormap.from_list('seq_blue', BLUES)

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE,
    'axes.edgecolor': '#dedcd3', 'axes.linewidth': 0.8,
    'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'xtick.labelcolor': INK2, 'ytick.labelcolor': INK2,
    'grid.color': '#eceae1', 'grid.linewidth': 0.8,
    'axes.grid': True, 'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'figure.dpi': 110, 'legend.frameon': False,
})

def finish(ax, title=None, sub=None, xlabel=None, ylabel=None):
    if title: ax.set_title(title, loc='left', pad=18 if sub else 8)
    if sub:   ax.text(0, 1.03, sub, transform=ax.transAxes, color=INK2, fontsize=9, va='bottom')
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    return ax

## Load a trained model

Point `RUN` at any per-seed directory under `results_proto/`. The default picks the
first finished push run.

In [ ]:
candidates = sorted(glob.glob('results_proto/Mutagenicity/*/*/[0-9]*'))
candidates = [c for c in candidates if os.path.exists(os.path.join(c, 'best.pt'))]
assert candidates, 'no finished run found under results_proto/'
RUN = candidates[0]
print('loading:', RUN)

model = torch.load(os.path.join(RUN, 'best.pt'), map_location=DEVICE, weights_only=False)
model.eval()
split = pickle.load(open(os.path.join(RUN, 'data.pkl'), 'rb'))
train_ds = split['train_dataset']

L  = len(model.convs)
H  = model.hidden_dim
K  = model.proto_layers[0].num_prototypes
print(f'variant={type(model).__name__}  layers={L}  hidden={H}  prototypes={K}  task={model.task}')
print('phi_in.tau per conv:', [c.nn[0].phi_in.tau for c in model.convs], '(set to 10 during the end-to-end phase)')

In [ ]:
BATCH = 256
batch = next(iter(DataLoader(train_ds, batch_size=BATCH, shuffle=False))).to(DEVICE)
N, G = batch.num_nodes, batch.num_graphs
y = batch.y.reshape(-1)
print(f'batch: {G} graphs, {N} nodes')
print(f'labels: class0 (mutagen) {int((y==0).sum())}, class1 (non-mutagen) {int((y==1).sum())}')

## Tapping the real model

Everything below is read out of the **actual modules** with forward hooks - nothing
about the forward pass is re-implemented here. One real `model(...)` call populates
every stage.

The only formula reproduced by hand is `phi`'s argument `w*x + b`, and it is built
from the module's own `w` and `b` (the step is applied inside `Phi.forward`, so
there is no sub-module to hook).

In [ ]:
class Taps:
    '''Capture module inputs/outputs from the real forward pass.'''
    def __init__(self, model): self.model, self.h, self._hs = model, {}, []
    def __enter__(self):
        put = lambda key: (lambda *a: self.h.__setitem__(key, (a[1][0] if len(a)==2 else a[2]).detach()))
        for i, c in enumerate(self.model.convs):
            ll = c.nn[0]
            # pre-hook on c.nn -> exactly what GINConv feeds the LogicalLayer (the aggregate)
            self._hs.append(c.nn.register_forward_pre_hook(
                lambda mod, inp, i=i: self.h.__setitem__(f'agg{i}', inp[0].detach())))
            self._hs.append(ll.phi_in.register_forward_hook(
                lambda mod, inp, out, i=i: self.h.__setitem__(f'lit{i}', out.detach())))
            self._hs.append(c.register_forward_hook(
                lambda mod, inp, out, i=i: self.h.__setitem__(f'h{i}', out.detach())))
        self._hs.append(self.model.fc.register_forward_pre_hook(
            lambda mod, inp: self.h.__setitem__('fc_in', inp[0].detach())))
        return self
    def __exit__(self, *a):
        for x in self._hs: x.remove()
        self._hs = []

with Taps(model) as taps, torch.no_grad():
    logits = model(batch.x.float(), batch.edge_index, batch.batch)

T  = taps.h
xs = [T[f'h{i}'] for i in range(L)]           # per-layer node states, as the model produced them
print('captured from the real forward pass:')
for key in [f'agg{i}' for i in range(L)] + [f'lit{i}' for i in range(L)] + [f'h{i}' for i in range(L)] + ['fc_in']:
    print(f'  {key:<7} {tuple(T[key].shape)}')
print(f'  logits  {tuple(logits.shape)}')

## Stage 0-1 - the input, and its negation

Each node is a 14-way atom one-hot. The layer is fed `[x, 1-x]`: every literal
alongside its complement, because the logic layer's weights are non-negative and so
cannot negate anything on its own.

In [ ]:
x0 = batch.x.float()
x1 = torch.hstack([x0, 1 - x0])
print(f'stage 0  x        {tuple(x0.shape)}   values {sorted(set(x0.unique().tolist()))}')
print(f'stage 1  [x,1-x]  {tuple(x1.shape)}')

counts = x0.argmax(-1).bincount(minlength=14).cpu().numpy()
order  = np.argsort(-counts)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar([ATOMS[i] for i in order], counts[order], color=SERIES[0], width=0.62)
for i, v in zip(range(len(order)), counts[order]):
    if v > 0: ax.text(i, v, f' {v}', ha='center', va='bottom', fontsize=8, color=INK2)
ax.set_yscale('log'); ax.grid(axis='x', visible=False)
finish(ax, 'Atom composition of the batch', f'{N} nodes across {G} molecules', ylabel='nodes (log)')
plt.tight_layout(); plt.show()

## Stage 2 - aggregation produces **counts**, not truth values

`GINConv` computes `(1+eps)*x_v + sum over neighbours`, and **only then** applies the
logic layer. So what `phi_in` actually sees is a count over the closed neighbourhood.

Note `eps = 0`: your own contribution is added into the *same* count as a neighbour's,
so no predicate can distinguish "I am a carbon" from "a neighbour is a carbon".

In [ ]:
conv0 = model.convs[0]
print('eps =', float(conv0.eps), '-> centre node is indistinguishable from a neighbour' if float(conv0.eps)==0 else '')

agg = T['agg0']          # tapped straight off GINConv -> LogicalLayer, not recomputed
print(f'stage 2  aggregate {tuple(agg.shape)}   range [{agg.min():.0f}, {agg.max():.0f}]')

fig, ax = plt.subplots(figsize=(7, 3.2))
vals, bins = np.histogram(agg.detach().cpu().numpy().ravel(), bins=np.arange(-0.5, agg.max().item()+1.5))
ax.bar(bins[:-1]+0.5, vals, color=SERIES[0], width=0.72)
ax.grid(axis='x', visible=False)
finish(ax, 'What the logic layer actually receives', 'neighbourhood counts, not 0/1 - layer 0',
       xlabel='count over closed neighbourhood', ylabel='occurrences')
plt.tight_layout(); plt.show()

## Stage 3 - the step function, and where it wraps

`phi_in` computes `step(w * count + b)`. The name suggests a threshold at `t = -b/w`,
but `step` is a **truncated Fourier square wave**, so it is *periodic with period 2pi* -
monotone only on the first half-period.

The plot below draws the actual `step` function over the range of arguments this layer
produces, with the argument distribution underneath. **Anything past `pi` has wrapped**,
and for those units "count >= t" is the wrong reading - the predicate is modular.

In [ ]:
from models_proto.tell import step

phi = conv0.nn[0].phi_in
arg = (phi.w * agg + phi.b).detach()
frac_wrapped = ((arg < -math.pi) | (arg > math.pi)).float().mean().item()
print(f'stage 3  phi argument {tuple(arg.shape)}   range [{arg.min():.2f}, {arg.max():.2f}]')
print(f'         outside the monotone window (-pi, pi): {100*frac_wrapped:.1f}% of activations')

lo, hi = float(arg.min())-1, float(arg.max())+1
grid = torch.linspace(lo, hi, 2000)
tau = phi.tau if phi.tau is not None else 0
curve = step(grid, tau=tau).cpu().numpy()

fig, (ax, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True,
                              gridspec_kw={'height_ratios':[2.2,1], 'hspace':0.25})
ax.plot(grid.numpy(), curve, color=SERIES[0], lw=2)
for k in range(0, int(hi/math.pi)+1):
    ax.axvline(k*math.pi, color=MUTED, lw=0.8, ls=':', zorder=0)
ax.axvspan(-math.pi, math.pi, color='#eceae1', zorder=0)
ax.set_ylim(-0.08, 1.12); ax.grid(axis='x', visible=False)
# single series -> the title names it; direct-label the band instead of a legend box
ax.text(0, 0.5, 'monotone window\n(threshold reading valid here)', ha='center', va='center',
        fontsize=9, color=INK2)
# park the series label on a plateau, clear of the rising edges
ax.text(2.5*math.pi, 0.5, f'step(x, tau={tau})', ha='center', va='center',
        fontsize=9, color=SERIES[0], fontweight='bold')
finish(ax, 'step() is a square wave, not a threshold',
       'every dotted line is another sign flip - the "count >= t" reading holds only inside the shaded band',
       ylabel='step(x)')

ax2.hist(arg.cpu().numpy().ravel(), bins=120, color=SERIES[1])
ax2.axvspan(-math.pi, math.pi, color='#eceae1', zorder=0)
ax2.grid(axis='x', visible=False)
finish(ax2, None, None, xlabel='phi argument  w*count + b', ylabel='nodes x dims')
ax2.text(0.99, 0.9, f'{100*frac_wrapped:.0f}% outside the window', transform=ax2.transAxes,
         ha='right', color=INK2, fontsize=9)
plt.tight_layout(); plt.show()

## Stage 4-5 - literals, then the layer's own predicates

After `phi_in` the values are (approximately) truth values, and the layer output is
`sigmoid(literals @ W.T + b)` with `W >= 0` - a **monotone threshold function** over
those literals. Each of the 64 output dimensions is one GML predicate.

"Crisp" below means within 0.02 of 0 or 1. The further it is from 100%, the more the
predicate reading is an approximation rather than a fact.

In [ ]:
crisp, rate = [], []
for i, h in enumerate(xs):                     # xs came from the hooks, not a re-run
    crisp.append((((h < 0.02) | (h > 0.98)).float().mean().item()) * 100)
    rate.append(((h > 0.5).float().mean().item()) * 100)
    lit = T[f'lit{i}']
    print(f'stage 4  layer {i}  literals {tuple(lit.shape)}  |  '
          f'stage 5  h {tuple(h.shape)}   crisp {crisp[-1]:5.1f}%   active {rate[-1]:4.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
lbl = [f'layer {i}' for i in range(L)]
axes[0].bar(lbl, crisp, color=SERIES[0], width=0.55)
axes[0].axhline(100, color=MUTED, lw=0.8, ls=':')
axes[0].set_ylim(0, 105); axes[0].grid(axis='x', visible=False)
for i, v in enumerate(crisp): axes[0].text(i, v, f' {v:.0f}%', ha='center', va='bottom', fontsize=9, color=INK2)
finish(axes[0], 'How Boolean is each layer?', 'within 0.02 of 0 or 1', ylabel='% of activations')

axes[1].bar(lbl, rate, color=SERIES[1], width=0.55)
axes[1].grid(axis='x', visible=False)
for i, v in enumerate(rate): axes[1].text(i, v, f' {v:.1f}%', ha='center', va='bottom', fontsize=9, color=INK2)
finish(axes[1], 'How often does a predicate fire?', 'fraction of (node, dim) pairs above 0.5', ylabel='% active')
plt.tight_layout(); plt.show()

In [ ]:
# the actual feature matrix - one row per node, one column per predicate
fig, axes = plt.subplots(1, L, figsize=(11, 3.4))
for i, ax in enumerate(axes):
    im = ax.imshow(xs[i][:120].detach().cpu().numpy(), aspect='auto', cmap=SEQ, vmin=0, vmax=1,
                   interpolation='nearest')
    ax.set_title(f'layer {i}', loc='left', color=INK, fontsize=10, fontweight='bold')
    ax.set_xlabel('predicate dim'); ax.grid(False)
    if i == 0: ax.set_ylabel('node')
cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02); cb.outline.set_visible(False)
cb.set_label('activation', color=INK2)
fig.suptitle('Feature matrix per layer  -  sparse and increasingly binary', x=0.125, y=1.06,
             ha='left', color=INK, fontsize=11, fontweight='bold')
plt.show()

## Stage 6-7 - jumping knowledge, then **pooling destroys the truth values**

The three layers are concatenated (`[N, 192]`), so dims 0-63 are radius-1 predicates,
64-127 radius-2, 128-191 radius-3.

Then pooling. `max_v phi(v)` is still an existential ("some node satisfies phi"), but
`mean_v phi(v)` is the **fraction** of nodes satisfying phi - a number, not a truth
value. This is the step that makes graph-level prototypes hard to read as logic.

In [ ]:
hcat = torch.hstack(xs)
z = model.prototype_inputs(xs, batch.batch)[0]
half = z.shape[1] // 2
print(f'stage 6  concat {tuple(hcat.shape)}  (radius 1|2|3 = dims 0-63 | 64-127 | 128-191)')
print(f'stage 7  pooled {tuple(z.shape)}  ops={model.pool_ops}')

def pct_bool(t): return 100 * (((t < 0.02) | (t > 0.98)).float().mean().item())
names = ['node-level\n(before pooling)', f'{model.pool_ops[0]}-pooled', f'{model.pool_ops[1]}-pooled']
vals  = [pct_bool(hcat), pct_bool(z[:, :half]), pct_bool(z[:, half:])]

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={'width_ratios':[1,1.3]})
ax.bar(names, vals, color=[SERIES[0], SERIES[1], SERIES[2]], width=0.55)
ax.set_ylim(0, 105); ax.grid(axis='x', visible=False)
for i, v in enumerate(vals): ax.text(i, v, f' {v:.0f}%', ha='center', va='bottom', fontsize=9, color=INK2)
finish(ax, 'Pooling turns truth values into fractions', 'share of entries within 0.02 of 0 or 1',
       ylabel='% Boolean')

ax2.hist(hcat.detach().cpu().numpy().ravel(), bins=60, color=SERIES[0], histtype='step', lw=2,
         label='node-level')
ax2.hist(z[:, :half].detach().cpu().numpy().ravel(), bins=60, color=SERIES[1], histtype='step', lw=2,
         label=f'{model.pool_ops[0]}-pooled')
ax2.set_yscale('log'); ax2.grid(axis='x', visible=False)
ax2.legend(fontsize=9, labelcolor=INK2)
finish(ax2, 'Where the mass goes', 'node-level piles at 0/1; pooled spreads across the interval',
       xlabel='value', ylabel='entries (log)')
plt.tight_layout(); plt.show()

## Stage 8 - prototype similarity

`s[g,k]` is the **fraction of the 384 dimensions** on which graph `g` agrees with
prototype `k`. That makes "matches prototype k" a *cardinality* statement ("enough
conditions hold"), not a conjunction ("all of these hold").

Two things to look at: how much `s` spreads across graphs (if it concentrates near 0.5,
the head has almost nothing to threshold), and whether the classes separate.

In [ ]:
with torch.no_grad():
    s = model.readout(xs, batch.batch)      # model's own readout, on the tapped layer states
print(f'stage 8  similarity {tuple(s.shape)}  range [{s.min():.3f}, {s.max():.3f}]')
print(f'         std across graphs: {s.std(0).mean().item():.4f}  (at init this is ~0.01)')

s_np = s.detach().cpu().numpy(); y_np = y.cpu().numpy()
sep = np.abs(s_np[y_np==0].mean(0) - s_np[y_np==1].mean(0))

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={'width_ratios':[1.2,1]})
bins = np.linspace(s_np.min(), s_np.max(), 45)
ax.hist(s_np[y_np==0].ravel(), bins=bins, color=SERIES[0], histtype='step', lw=2, label='mutagen')
ax.hist(s_np[y_np==1].ravel(), bins=bins, color=SERIES[1], histtype='step', lw=2, label='non-mutagen')
ax.grid(axis='x', visible=False); ax.legend(fontsize=9, labelcolor=INK2)
finish(ax, 'Similarity distribution by class', 'all prototypes pooled together',
       xlabel='similarity s', ylabel='graphs x prototypes')

o = np.argsort(-sep)
ax2.bar(range(K), sep[o], color=SERIES[2], width=0.62)
ax2.set_xticks(range(0, K, 2)); ax2.set_xticklabels([str(o[i]) for i in range(0, K, 2)])
ax2.grid(axis='x', visible=False)
finish(ax2, 'Which prototypes separate the classes', 'mean similarity gap between classes',
       xlabel='prototype (sorted)', ylabel='|class gap|')
plt.tight_layout(); plt.show()

## Stage 9 - the logic head, and which prototypes it actually uses

The head sees `[s, 1-s]`, passes it through its own `phi_in`, and combines with
non-negative weights. Summing the weight a prototype receives across both its literal
and its negation tells you how much the decision leans on it.

In [ ]:
out = logits                       # from the real forward pass above
lit = T['fc_in']                   # what the head's phi_in actually received: [s, 1-s]
W   = model.fc.weight.detach()
imp = (W[:, :K] + W[:, K:]).abs().sum(0).cpu().numpy()
used = (imp >= 0.1 * imp.max()).sum()
print(f'stage 9  head literals {tuple(lit.shape)} -> logits {tuple(out.shape)}')
print(f'         prototypes carrying >=10% of max weight: {used}/{K}')

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={'width_ratios':[1.3,1]})
o = np.argsort(-imp)
cols = [SERIES[0] if imp[k] >= 0.1*imp.max() else '#d8d6cc' for k in o]
ax.bar(range(K), imp[o], color=cols, width=0.62)
ax.set_xticks(range(K)); ax.set_xticklabels([str(k) for k in o], fontsize=8)
ax.grid(axis='x', visible=False)
ax.text(0.99, 0.92, f'{used} of {K} carry the decision', transform=ax.transAxes, ha='right',
        color=INK2, fontsize=9)
finish(ax, 'Head weight per prototype', 'grey = effectively unused (<10% of max)',
       xlabel='prototype', ylabel='summed |weight|')

p0 = out[y==0, 0].detach().cpu().numpy(); p1 = out[y==1, 0].detach().cpu().numpy()
ax2.hist(p0, bins=30, color=SERIES[0], histtype='step', lw=2, label='mutagen')
ax2.hist(p1, bins=30, color=SERIES[1], histtype='step', lw=2, label='non-mutagen')
ax2.grid(axis='x', visible=False); ax2.legend(fontsize=9, labelcolor=INK2)
finish(ax2, 'Output for class 0', 'the head is what has to separate these',
       xlabel='P(class 0)', ylabel='graphs')
plt.tight_layout(); plt.show()

# One molecule at a time

Everything above is aggregated over 256 graphs. This section takes **two individual
molecules** - one mutagen, one non-mutagen - and follows each through the same stages,
so you can see a single decision rather than a distribution.

In [ ]:
# pick one clearly-classified example of each class
pred = logits.argmax(-1)
conf = logits.max(-1).values
ok0 = ((y == 0) & (pred == 0)).nonzero().flatten()
ok1 = ((y == 1) & (pred == 1)).nonzero().flatten()
G0 = int(ok0[conf[ok0].argmax()])      # most confident correct mutagen
G1 = int(ok1[conf[ok1].argmax()])      # most confident correct non-mutagen
CASES = [(G0, 'mutagen'), (G1, 'non-mutagen')]

for gi, name in CASES:
    nodes = (batch.batch == gi).nonzero().flatten()
    atoms = batch.x[nodes].argmax(-1)
    comp = {}
    for a in atoms.tolist(): comp[ATOMS[a]] = comp.get(ATOMS[a], 0) + 1
    comp = ', '.join(f'{k}{v}' for k, v in sorted(comp.items(), key=lambda kv: -kv[1]))
    print(f'graph {gi:3d}  {name:12s}  {len(nodes):3d} atoms   {comp}')
    print(f'              P(mutagen)={logits[gi,0]:.3f}  P(non-mut)={logits[gi,1]:.3f}')

hcat = torch.hstack(xs)                              # [N x 192] before pooling
zb   = model.prototype_inputs(xs, batch.batch)[0]    # [G x 384] after pooling (model's own)
print(f'\nbefore pooling {tuple(hcat.shape)}  ->  after pooling {tuple(zb.shape)}')

### Which predicates fire, node by node

One row per atom, one column per predicate. This is the molecule's actual internal
representation - not a summary of it.

In [ ]:
fig, axes = plt.subplots(2, L, figsize=(11, 5.6))
for r, (gi, name) in enumerate(CASES):
    nodes = (batch.batch == gi).nonzero().flatten()
    labels = [ATOMS[a] for a in batch.x[nodes].argmax(-1).tolist()]
    for c in range(L):
        ax = axes[r, c]
        blk = xs[c][nodes].cpu().numpy()
        im = ax.imshow(blk, aspect='auto', cmap=SEQ, vmin=0, vmax=1, interpolation='nearest')
        ax.grid(False)
        if len(nodes) <= 30:
            ax.set_yticks(range(len(nodes))); ax.set_yticklabels(labels, fontsize=6)
        ax.set_title(f'{name} - layer {c}  ({(blk>0.5).sum()} firing)',
                     loc='left', fontsize=9, color=INK)
        if r == 1: ax.set_xlabel('predicate unit')
        if c == 0: ax.set_ylabel('atom')
plt.tight_layout()
cb = fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02); cb.outline.set_visible(False)
cb.set_label('activation', color=INK2)
plt.show()

### Before and after pooling, all dimensions, one graph

For a single molecule: on the left, every one of the 192 node-level dimensions - the
grey range is the spread across that molecule's atoms, and the two markers are what
`mean` and `max` pooling collapse it to. On the right, the same thing as distributions.

This is the whole 192 -> 384 step for one graph, with nothing averaged away.

In [ ]:
# driven by the model, not hardcoded: 'graph' pools mean|max, 'node'/'both' add sum
POOL_OPS = tuple(getattr(model, 'pool_ops', None) or model.node_pool_ops)
BOUNDED  = [op for op in POOL_OPS if op != 'sum']
HAS_SUM  = 'sum' in POOL_OPS
if HAS_SUM: print('note: sum is unbounded, so it gets its own panel (never a second y-axis)')

# WHAT gets pooled differs by variant:
#   'graph' pools the predicate vectors, THEN compares to prototypes
#   'node'  compares to prototypes FIRST, then pools the similarities
proto_in = model.prototype_inputs(xs, batch.batch)[0]
NODE_LEVEL = proto_in.shape[0] == batch.num_nodes
if NODE_LEVEL:
    with torch.no_grad(): PRE = model.proto_layers[0](proto_in)   # [N x K] similarities
    PRE_LABEL, PRE_UNIT = 'per-node similarity to each prototype', 'prototype'
else:
    PRE, PRE_LABEL, PRE_UNIT = hcat, 'per-node predicate vector', 'node-level dimension'
NDIM = PRE.shape[1]
print(f'this model pools {PRE_LABEL}  [{PRE.shape[0]} x {NDIM}]  with {POOL_OPS}')

for gi, name in CASES:
    nodes = (batch.batch == gi).nonzero().flatten()
    hg   = PRE[nodes]
    lo   = hg.min(0).values.cpu().numpy()
    # the model's OWN pooling, one op at a time
    pooled = {op: model._pool(PRE, batch.batch, (op,))[gi].cpu().numpy() for op in POOL_OPS}
    full = model._pool(PRE, batch.batch, POOL_OPS)[gi].cpu().numpy()
    assert np.allclose(np.concatenate([pooled[op] for op in POOL_OPS]), full, atol=1e-5)

    ncol = 2 + (1 if HAS_SUM else 0)
    widths = [2.1] + ([0.9] if HAS_SUM else []) + [1.0]
    fig, axs = plt.subplots(1, ncol, figsize=(12 + (2.4 if HAS_SUM else 0), 3.6),
                            gridspec_kw={'width_ratios': widths})
    axA, axB = axs[0], axs[-1]
    d = np.arange(NDIM)
    axA.vlines(d, lo, hg.max(0).values.cpu().numpy(), color='#d8d6cc', lw=1.1)
    for oi, op in enumerate(BOUNDED):
        axA.plot(d, pooled[op], ls='none', marker='_', ms=5, mew=1.6,
                 color=SERIES[oi], label=f'{op}-pooled')
    if not NODE_LEVEL:                      # layer boundaries only exist for predicate dims
        for b_ in range(1, L):
            axA.axvline(b_*H, color=MUTED, lw=0.8, ls=':')
            axA.text(b_*H, 1.04, f' layer {b_}', color=MUTED, fontsize=8, va='bottom')
        axA.text(0, 1.04, ' layer 0', color=MUTED, fontsize=8, va='bottom')
    axA.set_ylim(-0.04, 1.12); axA.set_xlim(-2, NDIM+1); axA.grid(axis='x', visible=False)
    # markers cover the whole plot area - park the legend just outside the top-right
    axA.legend(fontsize=9, labelcolor=INK2, loc='lower right', bbox_to_anchor=(1.0, 1.005), ncol=2)
    finish(axA, f'{name}  (graph {gi}, {len(nodes)} atoms)',
           'grey = spread across this molecule\'s atoms; markers = what pooling keeps',
           xlabel=PRE_UNIT, ylabel='value')

    # sum is unbounded -> its own panel rather than a second y-scale on axA
    if HAS_SUM:
        axS = axs[1]
        axS.plot(d, pooled['sum'], ls='none', marker='_', ms=5, mew=1.6, color=SERIES[2])
        axS.grid(axis='x', visible=False)
        finish(axS, 'sum-pooled', f'unbounded - max {pooled["sum"].max():.0f}',
               xlabel='dimension', ylabel='value')

    bins = np.linspace(0, 1, 41)
    axB.hist(hg.cpu().numpy().ravel(), bins=bins, histtype='step', lw=2, color=MUTED,
             density=True, label=f'per-node ({hg.numel()} values)')
    for oi, op in enumerate(BOUNDED):
        axB.hist(pooled[op], bins=bins, histtype='step', lw=2, color=SERIES[oi],
                 density=True, label=f'{op}-pooled')
    axB.set_yscale('log'); axB.grid(axis='x', visible=False)
    axB.legend(fontsize=8, labelcolor=INK2)
    finish(axB, 'Same values, as distributions', None, xlabel='value', ylabel='density (log)')
    plt.tight_layout(); plt.show()

    def frac01(a): return 100*((a < 0.02) | (a > 0.98)).mean()
    print(f'{name} (graph {gi}):')
    print(f'   before pooling : {hg.numel():5d} values   {frac01(hg.cpu().numpy()):5.1f}% at 0/1   mean {hg.mean():.3f}')
    for op in POOL_OPS:
        v = pooled[op]
        extra = '  [unbounded]' if op == 'sum' else f'{frac01(v):5.1f}% at 0/1'
        print(f'   {op+"-pooled":<14} : {len(v):5d} values   {extra}   mean {v.mean():.3f}')
    if 'max' in pooled and 'mean' in pooled:
        mx, mn = pooled['max'], pooled['mean']
        print(f'   dims where max=1 but mean<0.1 (one atom triggers it): {int(((mx>0.98)&(mn<0.1)).sum())}/{NDIM}')
    print()

### Its similarity to each prototype

The bar is this molecule; the marker is the population mean for that prototype. Where
the two diverge is where this molecule is unusual.

In [ ]:
s_np_all = s.detach().cpu().numpy()
pop = s_np_all.mean(0)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, (gi, name) in zip(axes, CASES):
    v = s_np_all[gi]
    ax.bar(range(K), v, color=SERIES[0], width=0.62, label='this molecule')
    ax.plot(range(K), pop, ls='none', marker='_', ms=11, mew=2, color=SERIES[1],
            label='population mean')
    ax.set_xticks(range(0, K, 2)); ax.grid(axis='x', visible=False)
    ax.set_xlabel('prototype')
    finish(ax, f'{name}  (graph {gi})', None)
axes[0].set_ylabel('similarity')
h, lb = axes[0].get_legend_handles_labels()
fig.legend(h, lb, fontsize=9, labelcolor=INK2, ncol=2, loc='lower center', bbox_to_anchor=(0.5, -0.08))
plt.tight_layout(); plt.show()

### Why: the dimensions that actually decided it

For the prototype the head leans on most, these are the dimensions carrying the
class signal - with each one's address decoded, what the prototype expects there, and
what **this molecule** actually has.

In [ ]:
def addr(d):
    pool = 'mean' if d < L*H else 'max'; r = d % (L*H)
    return pool, r // H, r % H

Wfc = model.fc.weight.detach()
impo = (Wfc[:, :K] + Wfc[:, K:]).abs().sum(0)
kbest = int(impo.argmax())
Pk = (model.proto_layers[0].proto_logits.detach() > 0).float()[kbest]

contrib = Pk * zb + (1 - Pk) * (1 - zb)                  # per-dim agreement, per graph
gapd = (contrib[y==0].mean(0) - contrib[y==1].mean(0)).abs()
top = torch.argsort(gapd, descending=True)[:8]

print(f'prototype {kbest} (head weight {impo[kbest]:.3f})\n')
hdr = f'{"dim":>4} {"address":>18} {"expects":>8} {"mutagen":>9} {"non-mut":>9}  reading'
print(hdr); print('-'*len(hdr))
for d in top.tolist():
    pool, l, u = addr(d)
    exp = 'TRUE' if Pk[d] > 0 else 'FALSE'
    v0, v1 = zb[CASES[0][0], d].item(), zb[CASES[1][0], d].item()
    reading = (f'EXISTS node with phi[L{l},u{u}]' if pool == 'max'
               else f'fraction of nodes with phi[L{l},u{u}]')
    print(f'{d:>4} {pool+"-pool L"+str(l)+" u"+str(u):>18} {exp:>8} {v0:>9.2f} {v1:>9.2f}  {reading}')
print(f'\nagreement with prototype {kbest} on these {len(top)} dims:')
for gi, name in CASES:
    print(f'  {name:12s}: {contrib[gi, top].mean().item():.3f}   (over all {2*L*H} dims: {contrib[gi].mean().item():.3f})')

## Prototype health

Two checks that decide whether the prototypes mean anything:

1. **Are they distinct?** Duplicates waste capacity and make the explanation redundant.
2. **Are they still the molecules they were pushed onto?** The push snaps each prototype
   to a real training graph, but training continues afterwards, so they drift. If the
   binary pattern no longer matches, "this looks like molecule X" is no longer literally true.

In [ ]:
P = (model.proto_layers[0].proto_logits.detach() > 0).float()
D = (P @ P.t() + (1-P) @ (1-P).t()) / P.shape[1]
off = ~torch.eye(K, dtype=torch.bool, device=P.device)
distinct = len(set(map(tuple, P.int().cpu().tolist())))
print(f'distinct prototype patterns: {distinct}/{K}')
print(f'mean pairwise agreement: {D[off].mean():.3f}  (0.5 = unrelated, 1.0 = identical)')

fig, ax = plt.subplots(figsize=(4.6, 4))
im = ax.imshow(D.cpu().numpy(), cmap=SEQ, vmin=0.5, vmax=1.0, interpolation='nearest')
ax.set_xticks(range(0, K, 2)); ax.set_yticks(range(0, K, 2)); ax.grid(False)
cb = fig.colorbar(im, ax=ax, fraction=0.046); cb.outline.set_visible(False)
cb.set_label('agreement', color=INK2)
finish(ax, 'Prototype pairwise agreement', f'{distinct}/{K} distinct patterns',
       xlabel='prototype', ylabel='prototype')
plt.tight_layout(); plt.show()

In [ ]:
# did the prototypes stay on the molecules the push selected?
push_log = os.path.join(RUN, 'push_log.json')
if os.path.exists(push_log) and json.load(open(push_log)):
    log = json.load(open(push_log)); last = log[-1]['indices'][0]
    feats = []
    with torch.no_grad():
        for b in DataLoader(train_ds, batch_size=128, shuffle=False):
            b = b.to(DEVICE); h = b.x.float(); hs = []
            for c in model.convs:
                h = c(torch.hstack([h, 1-h]), b.edge_index); hs.append(h)
            feats.append(model.prototype_inputs(hs, b.batch)[0])
    Zb = (torch.cat(feats) > 0.5).float()
    ag = np.array([(P[k] == Zb[last[k]]).float().mean().item() for k in range(K)])
    print(f'pushes: {len(log)} (epochs {log[0]["epoch"]}..{log[-1]["epoch"]})')
    print(f'prototypes EXACTLY equal to their pushed molecule: {(ag==1.0).sum()}/{K}')
    print(f'agreement with pushed molecule: min {ag.min():.3f}  mean {ag.mean():.3f}')

    drift = (1 - ag) * P.shape[1]        # dimensions that no longer match
    fig, ax = plt.subplots(figsize=(7.5, 3))
    ax.bar(range(K), drift, color=SERIES[0], width=0.62)
    ax.set_ylim(0, max(drift.max()*1.25, 1)); ax.grid(axis='x', visible=False)
    ax.set_xticks(range(0, K, 2))
    for k in range(K):
        ax.text(k, drift[k], f'{drift[k]:.0f}', ha='center', va='bottom', fontsize=8, color=INK2)
    ax.text(0.99, 0.92, '0 = still exactly the pushed molecule', transform=ax.transAxes,
            ha='right', color=INK2, fontsize=9)
    finish(ax, 'How far each prototype drifted since the last push',
           f'dimensions differing from the snapped molecule, out of {P.shape[1]}',
           xlabel='prototype', ylabel='dims that drifted')
    plt.tight_layout(); plt.show()
else:
    print('no pushes recorded (this run used --push_every 0)')

## Summary - the shapes, end to end

In [ ]:
rows = [
    ('0  raw input x',           tuple(x0.shape),   f'{len(ATOMS)}-way atom one-hot'),
    ('1  [x, 1-x]',              tuple(x1.shape),   'literal + negation'),
    ('2  GINConv aggregate',     tuple(agg.shape),  f'counts, eps={float(conv0.eps):.0f}'),
    ('3  phi argument',          tuple(arg.shape),  f'{100*frac_wrapped:.0f}% outside monotone window'),
    ('5  layer outputs',         tuple(xs[-1].shape), f'crisp {crisp[-1]:.0f}% at last layer'),
    ('6  jumping-knowledge',     tuple(hcat.shape), 'radius 1|2|3'),
    ('7  pooled z',              tuple(z.shape),    f'{pct_bool(z):.0f}% Boolean'),
    ('8  similarity s',          tuple(s.shape),    f'std {s.std(0).mean().item():.3f}'),
    ('9  logits',                tuple(out.shape),  f'{used}/{K} prototypes used'),
]
w = max(len(r[0]) for r in rows)
print(f'{"stage".ljust(w)}  {"shape".ljust(14)}  note')
print('-' * (w + 40))
for name, shape, note in rows:
    print(f'{name.ljust(w)}  {str(shape).ljust(14)}  {note}')